# Классификация ботов

In [1]:
import numpy as np
import pandas as pd
from ua_parser import parse

train = pd.read_csv('data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv('data/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv('data/events.csv.gz', parse_dates=['event_ts'])

print(train.shape, test.shape, events.shape)
print('доля ботов в train:', train.target.mean().round(4))
train[train['target'] == 0].head()

(11091, 5) (4909, 4) (328905, 14)
доля ботов в train: 0.0811


,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
0,ck_54a059eb7d3ea68b,2025-11-21 09:30:41,2026-04-06,2026-04-07,0
1,ck_7e4de46eeab82974,2025-09-23 10:10:24,2026-04-06,2026-04-07,0
2,ck_9320229ef6304522,2026-03-04 00:08:02,2026-04-06,2026-04-07,0
3,ck_30ccd25bc1714ed9,2026-04-05 10:52:40,2026-04-06,2026-04-07,0
4,ck_a77c5f05948cdeef,2026-01-01 03:54:35,2026-04-06,2026-04-07,0


In [2]:
events = events.drop_duplicates()

## Осмотреться

Прежде чем считать агрегаты, стоит посмотреть на данные: какие типы событий бывают, что
лежит в `platform` и `user_agent`, где пропуски, всё ли уникально.

In [3]:
print(events.event_name.value_counts(), '\n')
print(events.platform.value_counts(), '\n')
print(events.seller_type.value_counts(), '\n')
print('пропуски по колонкам:')
print(events.isna().mean().round(3))

event_name
item_view               118990
search_results_view      98929
photo_swipe              35962
favorite_add             18739
seller_page_view         17180
contact_phone_show       11154
captcha_shown             7831
login                     6178
contact_chat_open         6047
contact_message_sent      3032
Name: count, dtype: int64 

platform
desktop    46157
WEB        45799
Web        45703
web        45284
ANDROID    41809
Android    41637
android    41542
iOS         4035
iphone      4035
IOS         4028
ios         4013
Name: count, dtype: int64 

seller_type
private    110955
pro         81199
Name: count, dtype: int64 

пропуски по колонкам:
cookie_id        0.000
event_ts         0.000
eid              0.000
event_name       0.000
platform         0.000
user_agent       0.000
item_id          0.349
item_category    0.100
item_location    0.072
seller_type      0.407
search_query     0.695
search_page      0.695
pointer_x        0.670
pointer_y        0.670
dtype: 

## Окно наблюдения

Признаки считаем только по событиям внутри окна: `window_start_ts <= event_ts < window_end_ts`.
Это требование из условия, а не рекомендация.

In [4]:
def events_in_window(events, meta):
    ev = events.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id')
    return ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)]

ev_tr = events_in_window(events, train)
ev_te = events_in_window(events, test)
print(len(ev_tr), len(ev_te))

195484 88349


In [5]:
def family(raw):                #выбираем из User-agent семейство 
    if pd.notna(raw):
        parsed = parse(raw)
        if parsed.user_agent:   
            return parsed.user_agent.family
        else:
            return parsed.user_agent
    else:
        return None


ev_tr['UA_family'] = ev_tr.user_agent.apply(family)
ev_te['UA_family'] = ev_te.user_agent.apply(family)
ev_tr = ev_tr.sort_values(['event_ts'])                   # Сортируем события для дальнейших вычислений
ev_te = ev_te.sort_values(['event_ts'])
ev_tr['platform'] = ev_tr['platform'].apply(lambda x: x.lower())       #Приводим к единому виду устройства
ev_te['platform'] = ev_te['platform'].apply(lambda x: x.lower())
ev_tr.platform.unique()

<StringArray>
['desktop', 'web', 'android', 'iphone', 'ios']
Length: 5, dtype: str

<h1>Генерация признаков</h1>

In [6]:
def avg_time_diff(group, feature):                                # Среднее время между значениями признака event_ts
    res = group[feature].apply(lambda x: x.diff().mean())
    res = res.dt.total_seconds()
    res = res.apply(lambda x: round(x, 2))
    return res

def median_time_diff(group, feature):               # Медиана времени между event
    res = group[feature].apply(lambda x: x.diff().median())
    res = res.dt.total_seconds()
    res = res.apply(lambda x: round(x, 2))
    return res

def std_time_diff(group, feature):                  #стандартное отклонение разниц между event_ts      
    res = group[feature].apply(lambda x: x.diff().std())
    res = res.dt.total_seconds()
    res = res.apply(lambda x: round(x, 2))
    return res

def duration(group, feature):                          #Длительность сессии
    res = group[feature].apply(lambda x: x.max() - x.min())
    res = res.dt.total_seconds()
    res = res.apply(lambda x: round(x, 2))
    return res

def min_time_diff(group, feature):                      #Минимальная разница между events
    res = group[feature].apply(lambda x: x.diff().min())
    return res.dt.total_seconds()

In [7]:
def basic_features(ev, meta):                     # Формируем признаки для train и test выборок
    g = ev.groupby('cookie_id')
    f = pd.DataFrame({
        'item_nunique': g.item_id.nunique(),              # Количество уникальных обьявлений
        'event_nunique': g.event_name.nunique(),              # Количество уникальных событий cookie
        'UA_family': g.UA_family.agg(lambda x: next(iter(x.mode()), np.nan)),              # Наиболее частое семейство из User-agent 
        'location_nunique': g.item_location.nunique(),              # Количество уникальных локаций
        'category_nunique': g.item_category.nunique(),              # Количество уникальных категорий
        'search_query': g.search_query.nunique(),              # Количество уникальных поисковых запросов
        'std_time_diff': std_time_diff(g, 'event_ts'),              # см. одноименную функцию
        'pointer_nunique': g.pointer_x.nunique(),              # Количество уникальных значений указателя мыши по оси x
        'session_duration': duration(g, 'event_ts'),              # см. одноименную функцию
        'search_page_nunique': g.search_page.nunique(),              # Количество уникальных поисковых страниц
        'search_page_range': g.search_page.apply(lambda x: x.max() - x.min()),              # Размах значений поисковых страниц
        'activity_ratio': duration(g, 'event_ts') / (g.window_end_ts.first() - g.window_start_ts.first()).dt.total_seconds() ,              # Плотность активности пользователя
        'direct_view_share': g.event_name.apply(lambda x: sum(x == 'search_results_view')) / g.event_name.apply(lambda x: sum(x == 'item_view')),              # Отношение количество поисковых запросов к количеству просмотренных обявлений
        'min_time_diff': min_time_diff(g, 'event_ts'),              # см. одноименную функцию
        'contact_events_share': g.event_name.apply(lambda x: sum(x.isin({'contact_phone_show', 'contact_chat_open', 'contact_message_sent'}))) / g.event_name.count(),              # Доля событий связанных с контактами
        'swipes_per_view': g.event_name.apply(lambda x: sum(x == 'photo_swipe')) / g.event_name.apply(lambda x: sum(x == 'item_view')),              # Отношение свайпов фотографий к количеству просмотренных обьявлений
        'median_time_diff': median_time_diff(g, 'event_ts'),              # см. одноименную функцию
        'has_login': g.event_name.apply(lambda x: any(x == 'login')),              # Был ли login event у пользователя
        'avg_time_diff': avg_time_diff(g, 'event_ts')              # см. одноименную функцию
    })
    f = f.replace([np.inf, -np.inf], np.nan).fillna(0)
    f = meta[['cookie_id']].merge(f.reset_index(), on='cookie_id', how='left')
    return f.fillna(0)

Xtr = basic_features(ev_tr, train)
Xte = basic_features(ev_te, test)
ytr = train.target.values
Xtr.head()

,cookie_id,item_nunique,event_nunique,UA_family,location_nunique,category_nunique,search_query,std_time_diff,pointer_nunique,session_duration,search_page_nunique,search_page_range,activity_ratio,direct_view_share,min_time_diff,contact_events_share,swipes_per_view,median_time_diff,has_login,avg_time_diff
0,ck_54a059eb7d3ea68b,4,3,okhttp,5,3,3,29.95,0,167.0,3,5.0,0.001933,1.000000,1.0,0.000000,0.333333,21.0,False,27.83
1,ck_7e4de46eeab82974,19,7,Yandex Browser,12,1,5,2416.29,36,39476.0,6,7.0,0.456898,0.363636,2.0,0.024390,0.181818,57.5,True,986.90
2,ck_9320229ef6304522,19,9,Chrome,9,7,10,2306.45,35,36491.0,7,9.0,0.422350,1.857143,1.0,0.111111,0.428571,22.0,True,1042.60
3,ck_30ccd25bc1714ed9,10,5,Chrome,4,1,3,2502.08,0,17221.0,2,1.0,0.199317,0.214286,6.0,0.095238,0.000000,54.5,False,861.05
4,ck_a77c5f05948cdeef,16,6,Chrome,6,4,6,1748.44,27,19098.0,4,6.0,0.221042,0.500000,22.0,0.035714,0.333333,52.0,False,707.33


# Валидация

Тест лежит **позже** трейна по времени, поэтому и валидацию честно делать по времени, а не
случайным сплитом.

Метрику берём из `metric.py` — это ровно тот код, которым считает проверяющая система.
Своя реализация почти наверняка разойдётся с официальной на одинаковых `score`:
их нельзя разделять, группа равных значений отмечается целиком.

In [8]:
from catboost import CatBoostClassifier
from metric import precision_at_recall

is_valid = train.window_start_ts.ge('2026-04-17').values
cols = ['item_nunique', 'avg_time_diff', 'event_nunique', 'UA_family', 'location_nunique', 'std_time_diff',  'category_nunique', 'median_time_diff', 'pointer_nunique', 'session_duration',  'search_page_nunique', 'search_page_range', 'activity_ratio', 'direct_view_share', 'min_time_diff', 'contact_events_share', 'swipes_per_view', 'has_login', 'search_query']
seeds = [42, 41, 6, 21, 7, 9]
results_dates = []
results = []

dates = ['2026-04-14', '2026-04-15', '2026-04-16', '2026-04-17']

for date in dates:                                        # Валидацию провожу по 4 разным временным сплитам и по 6 random_seeds
    is_valid = train.window_start_ts.ge(date).values
    results = []
    for seed in seeds:
        model = CatBoostClassifier(iterations=750,
                                   depth=5,
                                   learning_rate=0.02,
                                   loss_function='Logloss',
                                   verbose=False,
                                   random_seed=seed
                                  )
        
        model.fit(Xtr.loc[~is_valid, cols], ytr[~is_valid], cat_features=['UA_family'])
        p_va = model.predict_proba(Xtr.loc[is_valid, cols])[:, 1]
        results.append(round(precision_at_recall(ytr[is_valid], p_va), 4))
    results_dates.append(results)
    
print([round(np.mean(i), 4) for i in results_dates])
print(round(np.mean([round(np.mean(i), 4) for i in results_dates]), 4))

[np.float64(0.5085), np.float64(0.4974), np.float64(0.5022), np.float64(0.551)]
0.5148


In [9]:
model = CatBoostClassifier(iterations=750,
                           depth=5,
                           learning_rate=0.02,
                           loss_function='Logloss',
                           verbose=False,
                           random_seed=6
                        )
    
model.fit(Xtr.loc[~is_valid, cols], ytr[~is_valid], cat_features=['UA_family'])
model.get_feature_importance(prettified=True)

,Feature Id,Importances
0,median_time_diff,18.241649
1,category_nunique,12.132818
2,location_nunique,10.956573
3,min_time_diff,10.038050
4,item_nunique,7.425685
5,contact_events_share,7.094287
6,search_page_range,5.043808
7,swipes_per_view,4.155453
8,search_page_nunique,3.517779
9,UA_family,3.309556


# Сабмит

In [10]:
model.fit(Xtr[cols], ytr, cat_features=['UA_family'])
sub = pd.DataFrame({
    'cookie_id': Xte.cookie_id,
    'score': model.predict_proba(Xte[cols])[:, 1],
})
assert len(sub) == len(test) and sub.score.between(0, 1).all()
sub.to_csv('submission.csv', index=False)
sub.head()

,cookie_id,score
0,ck_315fb710a0e371e7,0.003150
1,ck_a76ee3b3e3e522fd,0.149229
2,ck_94c9a4d382689e82,0.016402
3,ck_8eaf9509ad9462a0,0.009437
4,ck_9a88a5a989cb5bc6,0.015325
